# Prática de Clustering — Vértice Investimentos
**Aluno:** Gabriel Koji Kondo — 3°I  
**Disciplina:** Ciência de Dados  
**Entrega 1:** Entendimento do problema · EDA inicial · Plano de modelagem

---
## 1. Entendimento do Problema

### O que o Rafael está pedindo?

O Head de Relacionamento da Vértice percebeu que tratar 500 clientes com a mesma abordagem está gerando atendimento genérico e ineficiente. Ele quer saber **quem são os clientes de verdade** — não pelo cadastro, mas pelo comportamento real de investimento.

Traduzindo em termos analíticos:

> **Hipótese:** É possível identificar, a partir do comportamento observado nas carteiras, grupos de clientes com perfis de investimento distintos — e esses grupos devem divergir da classificação de risco autodeclarada (conservador / moderado / arrojado), evidenciando que declaração e comportamento real nem sempre coincidem.

### Por que é um problema de aprendizado não supervisionado?

Porque **não existe um rótulo correto previamente definido** para os perfis comportamentais. O `perfil_declarado_pelo_cliente` é o que o cliente *disse* sobre si mesmo — não uma verdade validada sobre como ele realmente investe. Não há um gabarito de "grupo correto" para cada cliente.

Queremos que o modelo **descubra padrões nos dados por conta própria**, sem supervisão humana. Isso é, por definição, clustering — uma técnica de aprendizado não supervisionado.

### Decisão analítica: `perfil_declarado_pelo_cliente`

Esta variável **não será usada como feature no modelo de clustering**.  
O motivo é direto: incluí-la contaminaria o modelo com exatamente a informação que queremos questionar. Queremos descobrir os grupos pelo comportamento real e, *depois*, comparar com o que foi declarado — esse cruzamento é a resposta à hipótese do Rafael.

`id_cliente` também será excluído: é apenas um identificador, sem valor informacional para o modelo.

---
## 2. Imports e Carregamento dos Dados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Estilo visual consistente
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("carteiras_clientes.csv", encoding="ISO-8859-1")

print(f"Shape: {df.shape}")
print(f"\nTipos de dados:")
print(df.dtypes)
print(f"\nNulos por coluna:")
print(df.isnull().sum())

---
## 3. Análise Exploratória dos Dados (EDA)

### 3.1 Estatísticas descritivas gerais

Antes de qualquer gráfico, vale entender a amplitude dos dados numericamente.

In [ ]:
df.describe().round(2)

**Observações iniciais:**

- `patrimonio_declarado_R$` varia de R\$ 8 mil a R\$ 2,3 milhões — amplitude brutal.  A mediana (~R\$ 492 mil) é bem menor que a média (~R\$ 597 mil), indicando assimetria à direita e presença de outliers.
- `operacoes_por_mes` tem mínimo 0 e máximo 56 — há clientes completamente inativos e outros operando quase todo dia útil.
- `prazo_medio_posicao_dias` vai de 3 a 927 dias — perfis de day trader a buy-and-hold extremo convivem na mesma base.
- A **soma dos percentuais de alocação** (`perc_*`) fica consistentemente em ~100% para todos os clientes, confirmando que os dados de carteira são coerentes.

### 3.2 Distribuição dos Perfis Declarados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem absoluta
order = ["conservador", "moderado", "arrojado"]
counts = df["perfil_declarado_pelo_cliente"].value_counts()[order]
axes[0].bar(order, counts, color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Contagem de clientes por perfil declarado")
axes[0].set_xlabel("Perfil")
axes[0].set_ylabel("Quantidade")
for i, v in enumerate(counts):
    axes[0].text(i, v + 2, str(v), ha="center", fontweight="bold")

# Percentual
pct = counts / counts.sum() * 100
axes[1].pie(pct, labels=order, autopct="%1.1f%%",
            colors=["#4C72B0", "#DD8452", "#55A868"], startangle=90)
axes[1].set_title("Proporção por perfil declarado")

plt.tight_layout()
plt.show()

print(counts.to_string())

A base é relativamente equilibrada: **35,6% conservador, 38,4% moderado, 26% arrojado**. Nenhum perfil é raro, o que facilita a análise comparativa posterior.

### 3.3 Distribuições das Variáveis Numéricas

In [ ]:
num_cols = [
    "idade", "anos_como_cliente", "patrimonio_declarado_R$", "valor_total_investido_R$",
    "perc_renda_fixa", "perc_acoes", "perc_fiis", "perc_crypto", "perc_exterior",
    "qtd_ativos_distintos", "operacoes_por_mes", "prazo_medio_posicao_dias", "maior_posicao_pct_carteira"
]

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, edgecolor="white", color="#4C72B0", alpha=0.85)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("frequência", fontsize=8)

# Esconder eixos extras
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuições das variáveis numéricas", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**O que observar nas distribuições:**

- **`patrimonio_declarado_R$` e `valor_total_investido_R$`**: fortemente assimétricas à direita. A maioria dos clientes tem patrimônio relativamente pequeno, mas poucos clientes com valores muito altos puxam a média para cima. Essas variáveis precisarão de normalização antes do clustering, ou os centróides serão dominados pela escala dessas colunas.

- **`perc_renda_fixa`**: distribuição bimodal — há um grupo concentrado perto de 0% e outro perto de 75–90%. Isso já sugere que existem perfis muito distintos de alocação na base.

- **`operacoes_por_mes`**: extremamente concentrada em valores baixos (0–5), com uma cauda longa de traders ativos. Dos 500 clientes, 60 não fizeram nenhuma operação no período, enquanto 98 fizeram mais de 10. Isso representa comportamentos opostos na mesma base.

- **`prazo_medio_posicao_dias`**: distribuição muito espalhada (3 a 927 dias). Há clientes com perfil de altíssima rotatividade (prazo < 30 dias) e outros que mantêm posições por mais de um ano.

### 3.4 Identificação de Outliers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

outlier_vars = [
    ("patrimonio_declarado_R$", "Patrimônio Declarado (R$)"),
    ("operacoes_por_mes", "Operações por Mês"),
    ("prazo_medio_posicao_dias", "Prazo Médio Posição (dias)"),
    ("qtd_ativos_distintos", "Qtd. Ativos Distintos"),
]

for ax, (col, label) in zip(axes, outlier_vars):
    ax.boxplot(df[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor="#AEC6CF"), medianprops=dict(color="black", linewidth=2))
    ax.set_title(label, fontsize=10)
    ax.set_ylabel(col, fontsize=8)

plt.suptitle("Boxplots — variáveis com maior potencial de outlier", fontsize=12)
plt.tight_layout()
plt.show()

# Quantificar outliers via IQR
print("Outliers detectados (critério IQR):\n")
for col, label in outlier_vars:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f"  {label}: {n_out} outliers  (Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f})")

**Impacto dos outliers no clustering:**  
O K-Means é sensível a outliers — eles puxam os centróides e podem criar clusters artificiais para acomodar poucos clientes extremos. Os 12 clientes com patrimônio acima de R\$ 1,9M, por exemplo, poderiam formar um cluster próprio apenas por escala financeira, sem que isso represente um comportamento de investimento diferente.

**Estratégia adotada:** normalizar os dados antes do modelo (StandardScaler) reduz o impacto da escala, mas não elimina o problema. Na etapa de modelagem, avaliaremos se esses clientes formam um cluster coerente ou se distorcem os demais.

### 3.5 Divergência Entre Perfil Declarado e Comportamento Real

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
perfil_order = ["conservador", "moderado", "arrojado"]
colors = ["#4C72B0", "#DD8452", "#55A868"]

vars_analise = [
    ("perc_renda_fixa", "% Renda Fixa"),
    ("perc_acoes", "% Ações"),
    ("operacoes_por_mes", "Operações por Mês"),
]

for ax, (col, label) in zip(axes, vars_analise):
    data = [df[df["perfil_declarado_pelo_cliente"] == p][col].values for p in perfil_order]
    bp = ax.boxplot(data, patch_artist=True, labels=perfil_order)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for median in bp["medians"]:
        median.set(color="black", linewidth=2)
    ax.set_title(f"{label} por perfil declarado", fontsize=10)
    ax.set_ylabel(label, fontsize=9)

plt.suptitle("Comportamento real vs. perfil autodeclarado", fontsize=12)
plt.tight_layout()
plt.show()

# Números que embasam a hipótese do Rafael
total_cons = (df["perfil_declarado_pelo_cliente"] == "conservador").sum()
total_arr  = (df["perfil_declarado_pelo_cliente"] == "arrojado").sum()

cons_baixa_rf   = ((df["perfil_declarado_pelo_cliente"] == "conservador") & (df["perc_renda_fixa"] < 50)).sum()
cons_crypto     = ((df["perfil_declarado_pelo_cliente"] == "conservador") & (df["perc_crypto"] > 10)).sum()
cons_acoes      = ((df["perfil_declarado_pelo_cliente"] == "conservador") & (df["perc_acoes"] > 30)).sum()
arr_alta_rf     = ((df["perfil_declarado_pelo_cliente"] == "arrojado") & (df["perc_renda_fixa"] > 70)).sum()

print("=== Divergências: declarado vs. comportamento ===\n")
print(f"Conservadores com < 50% em renda fixa:  {cons_baixa_rf}/{total_cons}  ({cons_baixa_rf/total_cons*100:.1f}%)")
print(f"Conservadores com > 10% em cripto:       {cons_crypto}/{total_cons}  ({cons_crypto/total_cons*100:.1f}%)")
print(f"Conservadores com > 30% em ações:        {cons_acoes}/{total_cons}  ({cons_acoes/total_cons*100:.1f}%)")
print(f"Arrojados com > 70% em renda fixa:       {arr_alta_rf}/{total_arr}  ({arr_alta_rf/total_arr*100:.1f}%)")

**A hipótese do Rafael se confirma nos dados:**

- **15,2%** dos conservadores têm menos da metade da carteira em renda fixa — para um perfil que supostamente prioriza segurança, isso é relevante.
- **4,5%** dos conservadores têm mais de 10% em cripto — ativos de altíssima volatilidade.
- **3,8%** dos arrojados têm mais de 70% em renda fixa — comportamento que contradiz o perfil declarado.

Esses números isolados podem parecer pequenos, mas indicam que o formulário de suitability *não captura* o comportamento real. O clustering irá revelar quão sistemática é essa divergência.

### 3.6 Mapa de Correlações

In [ ]:
num_cols_corr = [
    "idade", "anos_como_cliente", "patrimonio_declarado_R$", "valor_total_investido_R$",
    "perc_renda_fixa", "perc_acoes", "perc_fiis", "perc_crypto", "perc_exterior",
    "qtd_ativos_distintos", "operacoes_por_mes", "prazo_medio_posicao_dias", "maior_posicao_pct_carteira"
]

corr_matrix = df[num_cols_corr].corr()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, cbar_kws={"shrink": 0.8},
    annot_kws={"size": 8}
)
plt.title("Matriz de correlação — variáveis numéricas", fontsize=13, pad=15)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

**Correlações mais importantes para o modelo:**

| Par de variáveis | r | Interpretação |
|---|---|---|
| `patrimonio_declarado` × `valor_total_investido` | **+0.96** | Quase redundantes — manter apenas uma no modelo |
| `perc_renda_fixa` × `perc_acoes` | **-0.89** | Anticorrelação forte: são opostos na carteira |
| `perc_crypto` × `operacoes_por_mes` | **+0.61** | Quem tem crypto tende a operar mais |
| `qtd_ativos_distintos` × `maior_posicao_pct_carteira` | **-0.62** | Mais ativos → menor concentração (esperado) |
| `operacoes_por_mes` × `prazo_medio_posicao_dias` | **-0.51** | Quem opera mais, mantém posições menos tempo |
| `perc_renda_fixa` × `operacoes_por_mes` | **-0.59** | Quem tem muita renda fixa opera pouco |

A correlação altíssima entre `patrimonio_declarado_R$` e `valor_total_investido_R$` é um alerta: manter ambas no modelo pode dar peso duplo ao porte financeiro do cliente, distorcendo os clusters. Isso será endereçado no plano de modelagem.

### 3.7 Relação entre Variáveis de Alocação da Carteira

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pares = [
    ("perc_renda_fixa", "perc_acoes", "RF × Ações"),
    ("perc_acoes", "perc_crypto", "Ações × Cripto"),
    ("perc_fiis", "perc_exterior", "FIIs × Exterior"),
]

palette = {"conservador": "#4C72B0", "moderado": "#DD8452", "arrojado": "#55A868"}

for ax, (x, y, title) in zip(axes, pares):
    for perfil, color in palette.items():
        subset = df[df["perfil_declarado_pelo_cliente"] == perfil]
        ax.scatter(subset[x], subset[y], c=color, alpha=0.5, s=25, label=perfil)
    ax.set_xlabel(x, fontsize=9)
    ax.set_ylabel(y, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Relações entre variáveis de alocação (colorido por perfil declarado)", fontsize=12)
plt.tight_layout()
plt.show()

**Observação crítica:** No scatter RF × Ações, os três perfis se misturam consideravelmente. Não existe uma separação limpa entre conservador, moderado e arrojado — as fronteiras são difusas. Isso reforça visualmente que o perfil declarado não organiza os dados de forma clara, abrindo espaço para que o clustering encontre agrupamentos mais coerentes.

---
## 4. Plano de Modelagem

### 4.1 Variáveis selecionadas para o modelo

| Variável | Usar? | Justificativa |
|---|---|---|
| `id_cliente` | ❌ Remover | Identificador — sem valor informacional |
| `perfil_declarado_pelo_cliente` | ❌ Remover | É o que queremos questionar, não uma feature — usaremos depois para validação |
| `tem_assessor` | ❌ Remover | Variável estrutural (canal de atendimento), não descreve como o cliente investe |
| `patrimonio_declarado_R$` | ❌ Remover | Correlação de 0.96 com `valor_total_investido_R$` — redundante; manter a versão efetivamente investida é mais representativa do comportamento real |
| `idade` | ✅ Usar | Contexto demográfico relevante — jovens tendem a ter horizontes e tolerâncias diferentes |
| `anos_como_cliente` | ✅ Usar | Maturidade na corretora; clientes mais antigos tendem a ter comportamento mais consolidado |
| `valor_total_investido_R$` | ✅ Usar | Representa o porte real da carteira, não o declarado |
| `perc_renda_fixa` | ✅ Usar | Principal eixo de diferenciação de perfil |
| `perc_acoes` | ✅ Usar | Complementar à renda fixa; anticorrelação forte mas informações distintas |
| `perc_fiis` | ✅ Usar | Estratégia de renda passiva, diferencia perfis de longo prazo |
| `perc_crypto` | ✅ Usar | Alta correlação com comportamento ativo; diferencia perfis de risco |
| `perc_exterior` | ✅ Usar | Sofisticação da carteira e diversificação global |
| `qtd_ativos_distintos` | ✅ Usar | Nível de diversificação e sofisticação |
| `operacoes_por_mes` | ✅ Usar | Comportamento de trading vs. investimento passivo |
| `prazo_medio_posicao_dias` | ✅ Usar | Day trader vs. buy-and-hold — dimensão importante |
| `maior_posicao_pct_carteira` | ✅ Usar | Grau de concentração de risco |

**Total: 13 features para o modelo.**

---

### 4.2 Normalização

**Sim, normalização é obrigatória.**

O K-Means usa distância euclidiana para calcular a proximidade entre pontos. Sem normalização, variáveis com escalas grandes dominam o cálculo:

- `valor_total_investido_R$` varia de R\$ 2 mil a R\$ 2,2 milhões
- `operacoes_por_mes` varia de 0 a 56
- `perc_renda_fixa` varia de 0% a 97%

Sem normalização, um cliente que investe R\$ 100k a mais que outro seria considerado "mais distante" do que um cliente que tem 50% de diferença em alocação de ações — o que não faz sentido analítico.

**Método escolhido: `StandardScaler` (z-score)**  
Cada variável terá média 0 e desvio padrão 1 após a transformação. Preferido ao MinMaxScaler porque é menos sensível a outliers extremos.

---

### 4.3 Próximos passos (entrega final)

1. Aplicar `StandardScaler` nas 13 features selecionadas
2. Determinar K ideal via **Elbow Method** (inertia) e **Silhouette Score**
3. Treinar K-Means com o K escolhido
4. Nomear e descrever cada cluster em linguagem de negócio
5. Cruzar os clusters com `perfil_declarado_pelo_cliente` para responder a hipótese do Rafael
6. Redigir resposta ao e-mail

In [ ]:
# Preview do pré-processamento que será aplicado na entrega final
features = [
    "idade", "anos_como_cliente", "valor_total_investido_R$",
    "perc_renda_fixa", "perc_acoes", "perc_fiis", "perc_crypto", "perc_exterior",
    "qtd_ativos_distintos", "operacoes_por_mes", "prazo_medio_posicao_dias", "maior_posicao_pct_carteira"
]

X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape da matriz de features: {X.shape}")
print(f"\nApós StandardScaler:")
print(f"  Média por coluna (deve ser ~0): {X_scaled.mean(axis=0).round(3)}")
print(f"  Desvio padrão (deve ser ~1):   {X_scaled.std(axis=0).round(3)}")
print(f"\nDados prontos para K-Means — {X_scaled.shape[0]} clientes, {X_scaled.shape[1]} features.")